### Import thư viện

In [38]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

### Load các model đã train

In [39]:


lr_model = pickle.load(open(r"D:\UNIVERSITY\ĐACN\BI-Dashboard-Project\models\linear_model.pkl","rb"))
xgb_model = pickle.load(open(r"D:\UNIVERSITY\ĐACN\BI-Dashboard-Project\models\xgboost_model.pkl","rb"))
prophet_model = pickle.load(open(r"D:\UNIVERSITY\ĐACN\BI-Dashboard-Project\models\prophet_model.pkl","rb"))

### Load dataset

In [40]:

df = pd.read_csv(r"D:\UNIVERSITY\ĐACN\BI-Dashboard-Project\data\live\sales_dashboard.csv")


### Xử lý dữ liệu thiếu

In [41]:
df = df.dropna()

### Chuyển cột thời gian

In [42]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

### Feature Engineering từ thời gian

In [43]:

df['year'] = df['order_purchase_timestamp'].dt.year
df['month'] = df['order_purchase_timestamp'].dt.month
df['day'] = df['order_purchase_timestamp'].dt.day
df['hour'] = df['order_purchase_timestamp'].dt.hour

df = df.drop(columns=['order_purchase_timestamp'])

### Encode dữ liệu categorical

In [44]:
le = LabelEncoder()

categorical_cols = [
    'order_status',
    'seller_id',
    'customer_unique_id',
    'customer_city',
    'customer_state',
    'product_category_name',
    'Category_VN',
    'product_id'
]

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

### Chọn feature và target

In [45]:
X = df.drop(columns=['payment_value','order_id'])
y = df['payment_value']

### Chia tập train và test

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Predict với Linear Regression

In [47]:
y_pred_lr = lr_model.predict(X_test)

### Predict với XGBoost

In [48]:

y_pred_xgb = xgb_model.predict(X_test)

### Đánh giá Linear Regression

In [49]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

### Đánh giá XGBoost

In [50]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

### Chuẩn bị dữ liệu cho Prophet

In [51]:
df_prophet = df[['year','month','day','payment_value']].copy()

df_prophet['ds'] = pd.to_datetime(df_prophet[['year','month','day']])
df_prophet['y'] = df_prophet['payment_value']

df_prophet = df_prophet[['ds','y']]

### Predict với Prophet

In [52]:
forecast = prophet_model.predict(df_prophet)

y_pred_prophet = forecast['yhat']

### Đánh giá Prophet

In [53]:
mae_prophet = mean_absolute_error(df_prophet['y'], y_pred_prophet)

rmse_prophet = np.sqrt(mean_squared_error(df_prophet['y'], y_pred_prophet))

r2_prophet = r2_score(df_prophet['y'], y_pred_prophet)

### Tạo bảng so sánh model

In [54]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "XGBoost", "Prophet"],
    "MAE": [mae_lr, mae_xgb, mae_prophet],
    "RMSE": [rmse_lr, rmse_xgb, rmse_prophet],
    "R2 Score": [r2_lr, r2_xgb, r2_prophet]
})

results

,Model,MAE,RMSE,R2 Score
0,Linear Regression,63.302723,168.940626,0.605949
1,XGBoost,51.919194,104.490217,0.849258
2,Prophet,131.682397,274.372316,-0.004045
